# HilbertSFC Performance Deep Dive

This notebook is a deep dive into the design constraints and implementation choices behind HilbertSFC. We start from Skilling-style Hilbert encoders and work our way toward the bit-plane tiled finite-state-machine-based kernels used in this repo.

We will cover:

1. Why vectorized NumPy and naive "JIT"-compiled approaches become memory/latency bound (unpackbits, strided bit-slices, cache-line efficiency, MLP)
2. Reformulating traversal as a finite state machine, building the 2D lookup tables, and implementing encode/decode kernels
3. Packing and tiling tricks that unlock ILP, vectorization, and gathers (state-independent LUTs, register packing, bit-plane tiling)
4. A comparison to the Rust crate *fast_hilbert*


In [35]:
import sys
from dataclasses import replace
from functools import cache
from pathlib import Path

import numba as nb
import numpy as np
from numba.core import config as nb_config

repo_root = Path.cwd().parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from bench.hilbert_bench import (  # noqa: E402
    HilbertBench,
    HilbertBenchConfig,
    HilbertImplementation,
)

## Benchmark (2D)

#### 2D Points - Random, `nbits=32`, `n=5,000,000`

| Implementation | ns/pt (enc) | ns/pt (dec) | Mpts/s (enc) | Mpts/s (dec) |
| --- | ---: | ---: | ---: | ---: |
| 🔥**hilbertsfc (multi-threaded)** | 0.41 | 0.48 | 2410.39 | 2084.98 |
| 🔥**hilbertsfc (Python)** | 1.38 | 1.59 | 726.68 | 629.52 |
| [fast_hilbert (Rust)](https://crates.io/crates/fast_hilbert) | 12.24 | 12.03 | 81.67 | 83.11 |
| [hilbert_2d (Rust)](https://crates.io/crates/hilbert_2d) | 121.23 | 101.34 | 8.25 | 9.87 |
| [hilbert-bytes (Python)](https://pypi.org/project/hilbert-bytes/) | 2997.51 | 2642.86 | 0.334 | 0.378 |
| [numpy-hilbert-curve (Python)](https://pypi.org/project/numpy-hilbert-curve/) | 7606.88 | 5075.58 | 0.131 | 0.197 |
| [hilbertcurve (Python)](https://pypi.org/project/hilbertcurve/) | 14355.76 | 10411.20 | 0.0697 | 0.0961 |

## Background: Skilling-style algorithms

Classic Hilbert curve formulations are recursive and geometric (rotations and reflections). While mathematically elegant, they are ill-suited when expressed in programming languages to then be run on modern CPUs. Their recursive nature leads to function call overhead, repeated branching, and poor instruction-level parallelism. Skilling [1] showed that a hilbert curve can be expressed as bit permutations with gray code transforms. The Python implementations *hilbertcurve*, *numpy-hilbert-curve*, and *hilbert-bytes* are all implementations of Skilling's algorithm.

[1] John Skilling, *Programming the Hilbert Curve*, 2004.

## Vectorizing Skilling's Algorithm with NumPy: A Cautionary Tale

The most popular but also the slowest of these implementations is [*hilbertcurve*](https://github.com/galtay/hilbertcurve). It is a pure Python implementation of Skilling's algorithm. Due to the considerable overhead of the Python interpreter on each individual operation, it should come to no surprise that it is the slowest of the bunch.

[*numpy-hilbert-curve*](https://github.com/PrincetonLIPS/numpy-hilbert-curve) addresses this by implementing the algorithm with NumPy. In this way, the outer loop over all points may be removed, instead applying each operation element-wise over all points with a single NumPy call. This massively reduces the total number of operations in Python. While it is not uncommon to see a speed up of 50-100× when moving from Python to vectorized NumPy, *numpy-hilbert-curve* receive a modest speed up of only 2×. 

Why is the improvement so modest? The NumPy implementation essentially trades Python interpreter overhead for enormous volumes of memory traffic, which ends up becoming the new bottleneck. In particular, we see that `np.unpackbits` is applied to the input, expanding the data by 8×. Then, over this expanded data, the algorithm applies a series of bitwise operations on strided slices in nested loops:
```Python
for bit in range(num_bits):
    for dim in range(num_dims):
```

Inside, we see formulations like:

```Python
gray[:,dim,bit+1:] = np.logical_xor(gray[:,dim,bit+1:], to_flip)
gray[:,0,bit+1:] = np.logical_xor(gray[:,0,bit+1:], to_flip)
```

Each iteration scans multiple huge array slices, writes to slices and allocates many temporaries, all with poor cache locality. The total number of passes over the data is approximately:

```
num_bits × num_dims × num_ops × full_expanded_dataset_scan
```
where `num_ops` is roughly 10 (slices).

To demonstrate the severity of the problem, consider the memory access pattern: `points[:, dim, bit]`. With `ndim=2`, `nbytes=4` (32-bit integers), and unpacked bits into `uint8` byte arrays, the stores and loads are strided by `nbytes × 8 × ndims = 64 bytes`. This is exactly the size of a cache line, which implies that each cache line contains only 1 bit of useful data! That is a cache line efficiency of only `1/(8 bits × 64 bytes) = 1/512 = 0.195%`. In other words, it is throwing away 99.8% of every memory transfer.

To make matters worse, if the working set increase beyond a certain size, it will spill into L3 cache and eventually DRAM, which massively excerabates the problem. The CPU will then spend most of its time waiting for data to arrive. It should come as no surprise that *numpy-hilbert-curve* is bottlenecked by memory bandwidth and thus achieves only a modest speedup over the pure Python implementation which has good cache locality.

Now, my goal here is not to convince you that *numpy-hilbert-curve* is a poor implementation per se. Although there are some obvious improvements like reusing slices, the implementation may still be considered elegant and idiomatic. This argument is simply to illustrate the common pitfalls of vectorized NumPy code that chains together many single operations over huge arrays.


## Numba JIT Compilation: Faster but Still Memory-Bound

[*Hilbert-bytes*](https://github.com/hafaio/hilbert-bytes) is a more recent implementation that expresses the same algorithm but compiles it with Numba. This results in again a roughly 2× speedup compared to the NumPy implementation. However, the algorithmic bottleneck of memory bandwidth remains, and thus the overall gains are limited.

Conceptually, the core execution pattern is still:
```Python
for byte in range(nbytes):           # byte loop
    for bit in range(8):              # bit loop  
        for dim in range(ndim):       # dim loop
            # ALL operations use points[:, dim, byte] 
            # The [:] broadcasts over ALL num points (point loop)
```
This is fundamentally a loop ordering and data locality problem that Numba nor the LLVM backend is going to solve automatically.
Inspection of the generated LLVM IR and assembly shows:

- Aggressive loop unrolling
- Minor auto-vectorization
- Heavy address computation per element

However, these transformations provide little real benefit. The hot loop still reduces to scalar byte loads and stores, each accompanied by expensive address arithmetic (`add` / `mul` / `getelementptr` chains). The apparent "lanes" created by unrolling are not actual SIMD lanes, it's simply duplicated scalar work that LLVM was not able to auto-vectorize.

Because loop iterations read and write overlapping slices of points, LLVM must conservatively assume pointer aliasing. This is likely an uinentended side effect from the choice to remove temporaries and operate directly on the input array. The consequences can be massive, since now it cannot:

- Hoist loads
- Reorder memory operations
- Fuse updates across iterations

The end result is a loop with poor memory-level parallelism (MLP), limiting the number of concurrent cache misses the core can issue. As a result, memory latency is insufficiently hidden and begins to directly throttle throughput, reducing the effective bandwidth.

In short, performance is limited by effective memory bandwidth, not compute throughput, and JIT compilation alone cannot change that constraint.

# Loop Reordering and Data Layout

It should be fairly clear by now that the loops should be reordered such that the points-loop (coordinates) is placed on the outermost level.

These coordinates and indices are ideally loaded from a structure-of-arrays (SoA) format, which allows for contiguous memory access and will help with vectorization. Any performance-oriented data structure should already follow this format.

For the 2D case, the ideal execution pattern for encoding each point is:
1. Load coordinate `x` and `y` into registers
2. Compute the Hilbert index `idx` (keeping all intermediate results in registers)
3. Store `idx`

The same applies to decoding, but in reverse. In this way, each point is processed independently with minimal memory traffic. Each coordinate is loaded once, and the resulting index is stored once.

We can express this in Python with Numba as:

In [36]:
@cache
def build_hilbert_encode_2d_batch_impl(hilbert_encode_2d, nbits):
    @nb.njit()
    def encode_2d_batch(xs, ys, out):
        n = xs.shape[0]
        for i in nb.prange(n):
            out[i] = hilbert_encode_2d(xs[i], ys[i], nbits)

    return encode_2d_batch

### Batch wrappers for scalar kernels

This is a wrapper function that takes any scalar `hilbert_encode_2d` function/kernel and applies it element-wise to arrays of points `(xs, ys)`, writing to an output array of Hilbert indices `out`. This allows us to easily create batch versions of any scalar encoding function.  We decorate the inner function with `@nb.njit()` to compile it with Numba. The function also accepts `nbits` as an argument to pass to the underlying scalar function. We can build a similar wrapper for decoding:

In [37]:
@cache
def build_hilbert_decode_2d_batch_impl(hilbert_decode_2d, nbits):
    @nb.njit()
    def decode_2d_batch(idx, out_xs, out_ys):
        n = idx.shape[0]
        for i in nb.prange(n):
            out_xs[i], out_ys[i] = hilbert_decode_2d(idx[i], nbits)

    return decode_2d_batch

# Reformulating Hilbert Traversal as a Finite State Machine

With the Skilling-style algorithms, coordinates are transformed iteratively bit-by-bit, follow by a global bit transpose, and Gray decoding to produce the final index. However, if we follow the information flow rather than the implementation details, we may observe that Hilbert traversal is in essence a Markovian process where at each bit level the traversal only depends on:

- orientation/state of the curve
- the current coordinate bits (which determine the sub-quadrant the point enters next)

Everything else in the Skilling's formulation exists to preserve or recover those two pieces of information in a dimension-agnostic way. Orientation is encoded implicitly inside the transformed coordinates. That is why the code is dominated by XORs, and conditional dimension swaps. This essentialy maintains the local reference frame of the curve as it traverses space. The global transpose at the end simply reorganizes the accumulated bits into traversal order, and the Gray decode converts that traversal representation into the final index representation.

If instead you track orientation explicitly, the picture simplifies dramatically. At each bit level you can read the coordinate bits directly, interpret them relative to the current orientation, emit the corresponding Hilbert bits, and update the orientation. More formally, this may be expressed as a (Mealy) finite state machine with a state transition function `next_state = T(state, b)`, and an output function `q = O(state, b)`, where `state` is the curve's orientation, `b` the input coordinate bits at the current level, and `q` the output quadrant in Hilbert order.

With this finite state machine, we can express the encoding loop as follows:
```
state = initial_orientation
index = 0

for bit_level in (MSB to LSB):
    b = extract_coordinate_bits(x, y, bit_level)
    q = O(state, b)
    state = T(state, b)
    index = append(index, q)
```

The finite state machine can be expressed as lookup tables. Note that the geometric transforms disappear because they are precompiled into the transition tables. The (global) transpose disappears because traversal bits are emitted immediately. The Gray decode disappears because the emitted bits are already in the final representation. In short, the optimized encoder is essentially the Skilling algorithm with all temporary coordinate-space representations removed, leaving only the state transitions that actually define the Hilbert curve traversal.

# Building the Lookup Tables

We will build the lookup tables for the 2D case intuitively by identifying the 4 possible orientations (`state`) of the curve, and how they transition based on the input coordinate bits.

In [38]:
#
#
#           ╭───╮      ◄───╮      ▲   ╷      ╭───╴
#  state:   │ 0 │        1 │      │ 2 │      │ 3
#           ╵   ▼      ╶───╯      ╰───╯      ╰───►
#         ┌───┬───┐  ┌───┬───┐  ┌───┬───┐  ┌───┬───┐
#         │ 1 │ 2 │  │ 3 │ 2 │  │ 3 │ 0 │  │ 1 │ 0 │
#  q:  y↑ ├───┼───┤  ├───┼───┤  ├───┼───┤  ├───┼───┤
#         │ 0 │ 3 │  │ 0 │ 1 │  │ 2 │ 1 │  │ 2 │ 3 │
#         └───┴───┘  └───┴───┘  └───┴───┘  └───┴───┘
#            →x
#
# b = (x, y) bits packed into a 2-bit value
# q = LUT_2D_O_B[state, b]
#
# The LUT_2D_O_B is visualized through the 4 large cells representing the 4 possible states of the curve,
# and the small cells represent the 4 quadrants (q) within each state.

# ╭────╮    ╭────╮    ╭────╮    ╭────╮
# │ 0  │    │ 0  │    │ 0  │    │ 0  │
# │    ╰────╯    │    │    ╰────╯    │
# │     [0]      │    │      [0]     │
# ╰────╮    ╭────╯    ╰────╮    ╭────╯
#   1  │    │ 3         1  │    │ 3
# ╭────╯    ╰──────────────╯    ╰────╮
# │              [[0]]               │
# │    ╭─────────╮    ╭─────────╮    │
# │ 2  │      1  │    │ 3       │ 2  │
# ╰────╯    ╭────╯    ╰────╮    ╰────╯
#       [1] │              │ [3]
# ╭────╮    ╰────╮    ╭────╯    ╭────╮
# │ 0  │      1  │    │ 3       │ 0  │
# ╵    ╰─────────╯    ╰─────────╯    ▼

# In the drawing above, the third order Hilbert curve is drawn,
# which has 3 of the 4 state transitions across different levels.
# For example, we can see at the top level we are in state 0 (shown as [[0]]),
# then for b=2 (x=1, y=0) we transition to state 3 (shown as [3]).
# Then for state 3 and b=3 (x=1, y=1) we transition to state 2.


# The following lookup tables are derived from the above state machine diagram:
# fmt: off
LUT_2D_O_B = np.array(       # (state, b) -> q
    [[0, 1, 3, 2],
     [0, 3, 1, 2],
     [2, 3, 1, 0],
     [2, 1, 3, 0]]
    , dtype=np.uint8)

LUT_2D_T_B = np.array(    # (state, b) -> next_state
    [[1, 0, 3, 0],
     [0, 2, 1, 1],
     [2, 1, 2, 3],
     [3, 3, 0, 2]]
    , dtype=np.uint8)
# fmt: on

# Building the first finite state machine-based Hilbert encoder

We can use the pseudocode provided earlier to implement the 2D encode algorithm in a subset of Python that is supported by *nopython* Numba (eDSL). We simply right shift the coordinates starting with the most significant bit (MSB) to the least significant bit (LSB), and pack them together with a simple OR to form the input `b`. The output quadrant `q` is then appended to the index by shifting the index left by 2 bits and adding `q`. Finally, we update the state for the next iteration.

The start state is set to either 0 or 1 depending on `nbits` being even or odd. In this way, the Hilbert index obtained with a lower `nbits` is always the postfix of the result obtained with higher `nbits`, which is a nice property to have. The table above shows that when coordinates are prefixed with zeros (`b=00`), which is analagous to lower `nbits`, the state keeps jumping between 0 and 1 over the prefix.

In [39]:
@nb.njit(inline="always")
def hilbert_encode_2d(x, y, nbits):
    state = nbits % 2  # Start state is determined by nbits parity
    idx = 0
    # Process bits from MSB to LSB
    for bit in range(nbits - 1, -1, -1):
        x_bit = (x >> bit) & 0b1  # Extract the current bit of x
        y_bit = (y >> bit) & 0b1  # Extract the current bit of y
        b = (x_bit << 1) | y_bit  # Pack the current coordinate bits into a 2-bit value

        q = LUT_2D_O_B[state, b]  # Map (state, b) to Hilbert quadrant index
        idx = (idx << 2) | q  # Shift idx left by 2 bits and add q

        state = LUT_2D_T_B[state, b]  # Advance Hilbert orientation state

    return idx

# Reverse it for the Decoder

For the decoder we can simply reverse the process. Critically, our input is now the Hilbert quadrant index, and the output the coordinate bits. The state transitions remain the same, as `q` and `b` are simply two representations of the same position. Still, we would ideally have a state transition table that takes as input `q` instead of `b`. Otherwise we introduce an additional layer of indirection where the state transition look up needs to wait for the output `b` from the LUT_O table, which limits memory-level parallelism (MLP).

Since LUT_O_B is a bijection between `(state, b)` and `q`, we can simply enumerate all states and `b` values to invert the mapping and produce LUT_O_Q. The same holds for the transition table. We can do this programatically:

In [40]:
LUT_2D_O_Q = np.zeros((4, 4), dtype=np.uint8)  # (state, q) -> b
LUT_2D_T_Q = np.zeros((4, 4), dtype=np.uint8)  # (state, q) -> next_state

for state in range(4):
    for b in range(4):
        q = LUT_2D_O_B[state, b]
        next_state = LUT_2D_T_B[state, b]
        LUT_2D_O_Q[state, q] = b
        LUT_2D_T_Q[state, q] = next_state

print("LUT_2D_O_Q:\n", LUT_2D_O_Q)
print("LUT_2D_T_Q:\n", LUT_2D_T_Q)

LUT_2D_O_Q:
 [[0 1 3 2]
 [0 2 3 1]
 [3 2 0 1]
 [3 1 0 2]]
LUT_2D_T_Q:
 [[1 0 0 3]
 [0 1 1 2]
 [3 2 2 1]
 [2 3 3 0]]


### Decoder mechanics: extracting quadrants and bits

Since we have two dimension, each level is represented by 2 hilbert index bits. Therefore, we can extract the quadrant `q` at each level by right shifting the index by `2 * bit_level` and masking with `0b11`. This gives us the 2 bits corresponding to the current level's quadrant. We can then use LUT_O_Q to map `(state, q)` back to the coordinate bits `b`. The single coordinate bits can be easily extracted with a shift for `x` and mask for `y`. Note that `x` does not require a mask as the higher bit are already 0.

In [41]:
@nb.njit(inline="always")
def hilbert_decode_2d(idx, nbits):
    x = y = 0
    state = nbits % 2  # Start state is determined by nbits parity

    # Process Hilbert index from MSB to LSB
    for bit_shift in range(2 * (nbits - 1), -1, -2):
        q = (idx >> bit_shift) & 0b11

        b = LUT_2D_O_Q[state, q]  # Map (state, q) back to b

        x = (x << 1) | (b >> 1)  # 0bx0 --> 0b0x
        y = (y << 1) | (b & 1)  # 0bxy --> 0b0y

        state = LUT_2D_T_Q[state, q]  # Advance Hilbert orientation state

    return x, y

# Emperical validation for correctness and performance

We will use the Hilbert Bench harness suite to validate both the correctness and performance. The scalar kernels can be wrapped with the earlier defined batch functions. `nbits` will be provided as a compile time constant (Numba recognizes the closure variable and will inline it is literal values), which allows for many additional optimizations such as constant folding, full loop unrolling, dead code elimination, vectorization, constant shifts, and better instruction parallelism. Not all of these optimizations will be immediately realized, but we will be able to achieve all of them down the line with some careful engineering and code structuring. The only downside of this approach is that we need to generate separate kernels for each `nbits` value, but this is a small price to pay for the performance benefits; especially considering that a single reasonable upper bounds on `nbits` often suffices.

In [42]:
# Create the benchmark. We create one for 8, 16, and 32 bit coordinates.
cfg = HilbertBenchConfig(
    mode="random",
    ndim=2,
    nbits=32,
    n=5_000_000,
    trials=1,
    min_time_s=2,
    validate=True,
)

nbits_list = [8, 16, 28, 29, 30, 31, 32]
benches: dict[int, HilbertBench] = {}

for nbits in nbits_list:
    cfg_nbits = replace(cfg, nbits=nbits)
    bench = HilbertBench(cfg_nbits)
    benches[nbits] = bench

In [43]:
# Build implementation for benchmark
name = "hilbert_2d1b_2luts"


def enc_hilbert_2d1b_2luts(xs, ys, out, *, nbits):
    batch_impl = build_hilbert_encode_2d_batch_impl(hilbert_encode_2d, nbits)
    return batch_impl(xs, ys, out)


def dec_hilbert_2d1b_2luts(idx, out_xs, out_ys, *, nbits):
    batch_impl = build_hilbert_decode_2d_batch_impl(hilbert_decode_2d, nbits)
    return batch_impl(idx, out_xs, out_ys)


hilbert_2d1b_2luts_impl = HilbertImplementation(
    name=name, encode=enc_hilbert_2d1b_2luts, decode=dec_hilbert_2d1b_2luts
)

In [44]:
results = benches[32].run_one(hilbert_2d1b_2luts_impl, print_setup=False)

hilbert_2d1b_2luts/encode: 144.048 ms | 28.81 ns/pt | 34.71 ± 5.49 Mpts/s | trials=12
hilbert_2d1b_2luts/decode: 144.525 ms | 28.91 ns/pt | 34.60 ± 0.66 Mpts/s | trials=14


**Impressive!** We realised a 400× speedup over the *hilbertcurve* Python implementation, and 100× speedup over Numba JIT-compiled *hilbert-bytes*.

This is a massive improvement considering that the underlying algorithmic complexity is the same. But it should come as no surprise: it contains minimal bit manipulations, no recursion, no branching, and perhaps most important, excellent cache locality. The algorithm is also very compact and elegant, which also makes it easy to understand and verify.

To get a better understanding of where the performance improvements come from, we can inspect the generated LLVM IR.

In [ ]:
batch_enc = build_hilbert_encode_2d_batch_impl(hilbert_encode_2d, 32)
text = batch_enc.inspect_llvm(batch_enc.signatures[0])
print(text)

## Inspecting the LLVM IR (2-LUT kernel)

First of all, we may notice that the heap allocated lookup tables are inserted as global constants:
```
@.const.array.data = internal unnamed_addr constant [16 x i8] c"\00\01\03\02\00\03\01\02\02\03\01\00\02\01\03\00", align 1
@.const.array.data.1 = internal unnamed_addr constant [16 x i8] c"\01\00\03\00\00\02\01\01\02\01\02\03\03\03\00\02", align 1
```
This means they are stored in the read-only data section of the kernel binary.

Then in the main loop, we see a pattern that is repeated 4 times, which is the result from the compiler unrolling 4 iterations of the inner loop. The overall pattern looks as follows:

```
%0 = add i64 %lsr.iv, 3
...
%.374 = lshr i64 %.373, %0
%.378 = lshr i64 %.377, %0
%.379 = and i64 %.378, 1
%.375 = shl nuw nsw i64 %.374, 1
%.380 = and i64 %.375, 2
%.381 = or disjoint i64 %.379, %.380
%.458 = shl nuw nsw i64 %hilbert_encode_2d_v15_state.2.1219, 2
%.461 = or disjoint i64 %.381, %.458
%.462 = getelementptr i8, ptr @.const.array.data, i64 %.461
%.463 = load i8, ptr %.462, align 1
%.473 = zext i8 %.463 to i64
%.557 = getelementptr i8, ptr @.const.array.data.1, i64 %.461
%.558 = load i8, ptr %.557, align 1
%.566 = zext i8 %.558 to i64
...
%3 = shl i64 %hilbert_encode_2d_v15_idx.2.1220, 4
%4 = shl nuw nsw i64 %.473, 2
%.472.1 = or i64 %3, %4
```

The `%0 = add i64 %lsr.iv, 3` simply indicates that this is the first bit level in the 4 iterations (`bit + 3`) of the unrolled loop. Beyond that, the code can be relatively straightforwardly mapped back to the original source code. The `getelementptr` and `load` instructions correspond to the LUT lookups, and the shifts (`lshr` and `shl`), `and`s, and `or`s correspond to the bit manipulations to extract the coordinate bits and update the index and state.

We may observe several things from this snippet:

Fist of all, the loop is not fully unrolled. This is likely because it exceeds an instruction count threshold. However, as a result certain optimizations are left in the tank. Particularly, the bit level (`bit`) cannot be reduced to a compile time constant. This means that the shifts are variable, which prevents certain optimizations such as constant folding and better instruction scheduling. For example, `%.374 = lshr i64 %.373, %0` and `%.375 = shl nuw nsw i64 %.374, 1` can easily be combined into a single instruction if the loop was fully unrolled.

Furthermore, it stands out that the hot loop body is purely scalar, with no vectorized instructions. In fact, if we reduce `nbits` to 16, we see that the loop will be fully unrolled and that all instructions will be vectorized (except for look up table access). Interestingly, if we disable auto-vectorization the performance is even slightly better:

In [46]:
print("With loop vectorization enabled:")
nb_config.LOOP_VECTORIZE = 1  # type: ignore[reportAttributeAccessIssue]
build_hilbert_encode_2d_batch_impl.cache_clear()
benches[16].run_one(hilbert_2d1b_2luts_impl, print_setup=False)

print("With loop vectorization disabled:")
nb_config.LOOP_VECTORIZE = 0  # type: ignore[reportAttributeAccessIssue]
build_hilbert_encode_2d_batch_impl.cache_clear()
_ = benches[16].run_one(hilbert_2d1b_2luts_impl, print_setup=False)

nb_config.LOOP_VECTORIZE = 1  # type: ignore[reportAttributeAccessIssue]
# print(build_hilbert_encode_2d_batch_impl(hilbert_encode_2d, 16).signatures)

With loop vectorization enabled:
hilbert_2d1b_2luts/encode: 55.438 ms | 11.09 ns/pt | 90.19 ± 3.16 Mpts/s | trials=36
hilbert_2d1b_2luts/decode: 64.268 ms | 12.85 ns/pt | 77.80 ± 8.79 Mpts/s | trials=30
With loop vectorization disabled:
hilbert_2d1b_2luts/encode: 50.793 ms | 10.16 ns/pt | 98.44 ± 3.44 Mpts/s | trials=39
hilbert_2d1b_2luts/decode: 63.715 ms | 12.74 ns/pt | 78.47 ± 2.33 Mpts/s | trials=32


## 2-LUT → 1-LUT: reducing load pressure

This regression on vectorization is not ideal, but it is also not particularly surprising once we look at the microarchitectural constraints. The dominant bottleneck is not the raw instruction count, but how effectively the core can hide the latency of the `load` instructions.

Although the lookup tables are very small and therefore almost certainly resident in L1D cache, L1 hit latency is still ~4 cycles on most modern x86 cores (e.g., Intel Core and AMD Zen microarchitectures). If the two loads per bit cannot be overlapped, the cost becomes:
```
32 bits × 2 loads × 4 cycles ≈ 256 cycles ≈ 60 ns/point
```
This is not inherently catastrophic because an out-of-order (OoO) core can continue issuing independent µops while a load is pending. In particular, the bit packing required to construct `b` is effectively latency-hidden and therefore close to free in steady state.


A more structural optimization is to reduce load pressure entirely. Instead of performing two independent table lookups (one for next state and one for output quadrant), we can encode both into a single 4-bit value: `next_state << 2 | q`. This way, we only need one load per bit level instead of two. We can also flatten the lookup table so that we can index it with `state | b`.

Notably, if the stored `next_state` is pre-shifted by 2, it eliminates unnecessary shifts when computing the next index. There is no need to unpack via a right shift only to left shift for the subsequent iteration. Note that the original LLVM IR shows that the multi-dimensional indexing into the LUT is effectively flattened into a single index by performing `state << 2 | b` before the load:
```
%.458 = shl nuw nsw i64 %hilbert_encode_2d_v15_state.2.1219, 2
%.461 = or disjoint i64 %.381, %.458
```
The state shift can thus be eliminated in the new approach.

Encoding with one packed LUT:

In [47]:
# Each LUT cell contains 0b0000ssqq
# With ss the next state and qq the quadrant index for the current state and b.
LUT_2D_SB_SQ = np.zeros(4 * 4, dtype=np.uint8)

for state in range(4):
    for b in range(4):
        idx = (state << 2) | b
        q = int(LUT_2D_O_B[state, b])
        s_next = int(LUT_2D_T_B[state, b])
        LUT_2D_SB_SQ[idx] = (s_next << 2) | q


@nb.njit(inline="always")
def hilbert_encode_2d_1lut(x, y, nbits):
    state = idx = 0
    # Process bits from MSB to LSB
    for bit in range(nbits - 1, -1, -1):
        b = (((x >> bit) & 0b1) << 1) | ((y >> bit) & 0b1)
        sq = LUT_2D_SB_SQ[state | b]  # Map (state, b) to Hilbert quadrant index
        q = sq & 0b0011  # Extract q from the combined value
        idx = (idx << 2) | q
        state = sq & 0b1100  # Extract next state from the combined value
    return idx


name = "hilbert_2d1b_1lut"


def enc_hilbert_2d1b_1lut(xs, ys, out, *, nbits):
    batch_impl = build_hilbert_encode_2d_batch_impl(hilbert_encode_2d_1lut, nbits)
    return batch_impl(xs, ys, out)


hilbert_2d_1b_impl = HilbertImplementation(name=name, encode=enc_hilbert_2d1b_1lut)

In [48]:
results = benches[32].run_one(hilbert_2d_1b_impl, print_setup=False)

hilbert_2d1b_1lut/encode: 177.295 ms | 35.46 ns/pt | 28.20 ± 0.50 Mpts/s | trials=12


## Why the 1-LUT kernel regresses

The throughput regressed (from 30 ns/pt to 35 ns/pt) despite removing one load per bit level. Why?

The earlier 60 ns/point estimate assumes fully serialized loads. The fact that measured performance was already below this bound tells us that loads were overlapping in practice.

The real constraint is the dependency chain in the state update. Each iteration depends on the result of the previous load, preventing the next load from being issued early:
```
sq = LUT[idx_0]
state = sq & 0b1100
idx_1 = state | b
sq = LUT[idx_1]
```
On the critical path this becomes:
```
load -> and -> or -> load -> and -> or -> ...
```
Critical path per level ≈ 6 cycles:
```
4 cycles load
1 cycle AND
1 cycle OR
```
Across 32 levels:
```
32 × 6 cycles ≈ 192 cycles ≈ 40 ns/pt
```
40 ns/pt is closer to what we are getting in practice. Still, the measured throughput is slightly better than this estimate. This is because the outer points loop allows some degree of overlap, but this is still bounded by a finite reorder buffer (ROB) size.

## State-Independent LUT for Better Memory-Level Parallelism
It seems then that the loop-carried dependency on the state introduces a serialization that limits how effectively the core can hide load latency and achieve high instruction-level parallelism (ILP). However, while this state dependency is inherent to the algorithm and therefore looks like a hard bottleneck, the most expensive operation in the chain, the `load`, does not actually need to depend on `state`. We can simply pack all four input states into a single lookup value.

4 bits are needed for each `(state, q)` pair (`sq`). Since there are 4 possible states, we need 16 bits total to store all four `sq` values. Again, by storing the next state in the upper 2 bits, we can obtain the desired `sq` pair by doing:
```
LUT_2D_B_SQ[b] >> state
```
Here `state ∈ {0, 4, 8, 12}` (i.e., already pre-shifted), so we can directly shift by state without additional adjustments.

Note that the state evolution itself remains serialized, but the most expensive operation in the chain has been decoupled from that serialization. In a latency-bound loop such as this, that structural change is far more important than reducing instruction count.

In [ ]:
# ss: next_state bits
# qq: quadrant index bits

# Each LUT cell contains:
#           ssqq|ssqq|ssqq|ssqq
# state:    3    2    1    0
#
# Look up via: LUT_2D_B_SQ[b] >> state
#


def init_lut_2d_b_sq(dtype: type[np.unsignedinteger] = np.uint16):
    global LUT_2D_B_SQ
    LUT_2D_B_SQ = np.zeros(4, dtype=dtype)
    for state in range(4):
        for b in range(4):
            q = int(LUT_2D_O_B[state, b])
            s_next = int(LUT_2D_T_B[state, b])
            lane_shift = state << 2
            LUT_2D_B_SQ[b] |= ((s_next << 2) | q) << lane_shift


init_lut_2d_b_sq()


@nb.njit(inline="always")
def hilbert_encode_2d_state_independent(x, y, nbits):
    state = idx = 0
    # Process bits from MSB to LSB
    for bit in range(nbits - 1, -1, -1):
        b = ((x >> bit) & 0b1) << 1 | ((y >> bit) & 0b1)
        sq = LUT_2D_B_SQ[b] >> state  # Map (state, b) to Hilbert quadrant index
        q = sq & 0b0011
        idx = (idx << 2) | q
        state = sq & 0b1100
    return idx


name = "hilbert_2d1b_state_independent"


def enc_hilbert_2d1b_state_independent(xs, ys, out, *, nbits):
    batch_impl = build_hilbert_encode_2d_batch_impl(
        hilbert_encode_2d_state_independent, nbits
    )
    return batch_impl(xs, ys, out)


hilbert_2d1b_state_independent_impl = HilbertImplementation(
    name=name, encode=enc_hilbert_2d1b_state_independent
)

In [50]:
results = benches[32].run_one(hilbert_2d1b_state_independent_impl, print_setup=False)

hilbert_2d1b_state_independent/encode: 84.501 ms | 16.90 ns/pt | 59.17 ± 4.33 Mpts/s | trials=24


### Result: state-independent LUT speedup

The troughput is doubled by this optimization (35 ns/pt → 17 ns/pt) because it allows the core to issue multiple loads in parallel. The dependency chain now is:
```
sq = .. >> state
state = sq & 0b1100
```
which is only 2 cycles per level:
```
2 cycles × 32 levels = 64 cycles ≈ 15 ns/pt
```

# Register-Packed LUT
Now, you might have wondered: if there are also only 4 possible `b` values, why not pack those as well and just store the entire lookup into a single register of 64 bits?

Sure, let's do that!

In [51]:
# ss: next_state bits
# qq: quadrant index bits

# The register LUT:
#           ssqq|ssqq|ssqq|ssqq|ssqq|ssqq|ssqq|ssqq|ssqq|ssqq|ssqq|ssqq|ssqq|ssqq|ssqq|ssqq
# b:        3    2    1    0    3    2    1    0    3    2    1    0    3    2    1    0
# state:    3    3    3    3    2    2    2    2    1    1    1    1    0    0    0    0

LUT_2D_SQ = 0

for state in range(4):
    for b in range(4):
        q = int(LUT_2D_O_B[state, b])
        s_next = int(LUT_2D_T_B[state, b])

        idx = (state << 2) | b
        tbl_ptr = idx << 2

        LUT_2D_SQ |= ((s_next << 2) | q) << tbl_ptr

print(f"{LUT_2D_SQ=:016x}")  # 0x83DE_C97A_65B0_2F14


@nb.njit(inline="always")
def hilbert_encode_2d_register_lut(x, y, nbits):
    state = idx = 0
    lut_sq = np.uint64(0x83DE_C97A_65B0_2F14)
    # Process bits from MSB to LSB
    for bit in range(nbits - 1, -1, -1):
        b = ((x >> bit) & 0b1) << 1 | ((y >> bit) & 0b1)
        tbl_ptr = (state | b) << 2
        sq = lut_sq >> tbl_ptr  # sq = LUT[state, b]
        q = sq & 0b11
        idx = (idx << 2) | q
        state = sq & 0b1100
    return idx


name = "hilbert_2d1b_register_lut"


def enc_hilbert_2d1b_register_lut(xs, ys, out, *, nbits):
    batch_impl = build_hilbert_encode_2d_batch_impl(
        hilbert_encode_2d_register_lut, nbits
    )
    return batch_impl(xs, ys, out)


hilbert_2d1b_register_lut_impl = HilbertImplementation(
    name=name, encode=enc_hilbert_2d1b_register_lut
)

LUT_2D_SQ=83dec97a65b02f14


In [52]:
results = benches[32].run_one(hilbert_2d1b_register_lut_impl, print_setup=False)

hilbert_2d1b_register_lut/encode: 127.260 ms | 25.45 ns/pt | 39.29 ± 5.14 Mpts/s | trials=15


### Why register-packing regresses

The LUT code: `0x83DE_C97A_65B0_2F14` (in hexadecimal) essentially encodes the entire Hilbert curve traversal logic, and happens to fit perfectly in a single register of 64 bits.

While this looks appealing and removes memory accesses entirely, it regresses performance (though it still outperforms the original two-load LUT). In practice, we have traded out-of-order `load` execution for additional instructions to unpack the LUT value from the register. The critical dependency chain now is:
```
sb = state | ..
tbl_ptr = sb << 2
sq = .. >> tbl_ptr
state = sq & 0b1100
```
which is 4 cycles per level, up from 2 cycles before. The shifts and masks are now fully on the critical path, and unlike loads, they cannot overlap via memory-level parallelism. Although with full loop unrolling the compiler will constant fold the shifts, but then the dependency chain is still 3 cycles per level. 

# Achieving Full-Unroll and Vectorization
Given these results and oberservations, it makes sense to revert to the state-independent LUT with one load per bit level and continue optimizing from there.

Inspection of the emitted LLVM IR reveals that the hot loop consists purely of scalar instructions, alike the 2-LUT version. However, when `nbits` is reduced, e.g. to 16, the loop will be both fully unrolled and fully vectorized (except for look up table access). Unlike the 2-load LUT version, the kernel with the state-independent LUT benefits considerably from vectorization, achieving a 2× speedup:

In [53]:
print("With loop vectorization disabled:")
nb_config.LOOP_VECTORIZE = 0  # type: ignore[reportAttributeAccessIssue]
build_hilbert_encode_2d_batch_impl.cache_clear()
_ = benches[16].run_one(hilbert_2d1b_state_independent_impl, print_setup=False)

print("With loop vectorization enabled:")
init_lut_2d_b_sq()
nb_config.LOOP_VECTORIZE = 1  # type: ignore[reportAttributeAccessIssue]
build_hilbert_encode_2d_batch_impl.cache_clear()
benches[16].run_one(hilbert_2d1b_state_independent_impl, print_setup=False)

print("With loop vectorization enabled and 32-bit valued LUT:")
init_lut_2d_b_sq(dtype=np.uint32)
build_hilbert_encode_2d_batch_impl.cache_clear()
_ = benches[16].run_one(hilbert_2d1b_state_independent_impl, print_setup=False)

With loop vectorization disabled:
hilbert_2d1b_state_independent/encode: 34.838 ms | 6.97 ns/pt | 143.52 ± 16.64 Mpts/s | trials=55
With loop vectorization enabled:
hilbert_2d1b_state_independent/encode: 24.639 ms | 4.93 ns/pt | 202.93 ± 5.79 Mpts/s | trials=81
With loop vectorization enabled and 32-bit valued LUT:
hilbert_2d1b_state_independent/encode: 18.146 ms | 3.63 ns/pt | 275.55 ± 10.30 Mpts/s | trials=110


In [ ]:
batch_enc = build_hilbert_encode_2d_batch_impl(hilbert_encode_2d_state_independent, 16)
text = batch_enc.inspect_asm(batch_enc.signatures[0])
print(text)

## Quirks of Vectorization and Gather Instructions
Vectorization with the 16-bit LUT achieves a partial speed up from 7.30 ns/pt → 5.13 ns/pt. 
Examining the LLVM IR explains why. The vectorized load is scalarized, leading to a long sequence where each lane is extracted individually, followed by eight scalar `load i16` instructions, and then one-by-one repacking into a vector:

```
  %10 = extractelement <8 x i64> %9, i64 0
  %11 = getelementptr i16, ptr @.const.array.data, i64 %10
  %12 = extractelement <8 x i64> %9, i64 1
  %13 = getelementptr i16, ptr @.const.array.data, i64 %12
  %14 = extractelement <8 x i64> %9, i64 2
  %15 = getelementptr i16, ptr @.const.array.data, i64 %14
  %16 = extractelement <8 x i64> %9, i64 3
  %17 = getelementptr i16, ptr @.const.array.data, i64 %16
  %18 = extractelement <8 x i64> %9, i64 4
  %19 = getelementptr i16, ptr @.const.array.data, i64 %18
  %20 = extractelement <8 x i64> %9, i64 5
  %21 = getelementptr i16, ptr @.const.array.data, i64 %20
  %22 = extractelement <8 x i64> %9, i64 6
  %23 = getelementptr i16, ptr @.const.array.data, i64 %22
  %24 = extractelement <8 x i64> %9, i64 7
  %25 = getelementptr i16, ptr @.const.array.data, i64 %24
  %26 = load i16, ptr %11, align 2
  %27 = load i16, ptr %13, align 2
  %28 = load i16, ptr %15, align 2
  %29 = load i16, ptr %17, align 2
  %30 = load i16, ptr %19, align 2
  %31 = load i16, ptr %21, align 2
  %32 = load i16, ptr %23, align 2
  %33 = load i16, ptr %25, align 2
  %34 = insertelement <8 x i16> poison, i16 %26, i64 0
  %35 = insertelement <8 x i16> %34, i16 %27, i64 1
  %36 = insertelement <8 x i16> %35, i16 %28, i64 2
  %37 = insertelement <8 x i16> %36, i16 %29, i64 3
  %38 = insertelement <8 x i16> %37, i16 %30, i64 4
  %39 = insertelement <8 x i16> %38, i16 %31, i64 5
  %40 = insertelement <8 x i16> %39, i16 %32, i64 6
  %41 = insertelement <8 x i16> %40, i16 %33, i64 7
  %42 = zext <8 x i16> %41 to <8 x i32>
```

This is expected as the AVX2 target has no instruction for a vectorized gather of 16-bit values. 

One simple trick is to use a LUT with 32-bit elements. Then, indeed, LLVM emits a vectorized gather instruction:
```
%10 = getelementptr i32, ptr @.const.array.data, <8 x i64> %9
%wide.masked.gather = tail call <8 x i32> @llvm.masked.gather.v8i32.v8p0(<8 x ptr> %10, i32 4, <8 x i1> splat (i1 true), <8 x i32> poison)
```

which lowers to a single `vpgatherqd` instruction.

This yields the final speed up from 5.13 ns/pt → 3.68 ns/pt. 

### Differences between Intel and AMD
It should be noted that this speed up is mostly Intel specific (particularly *Skylake* and higher). AMD *Zen* microarchitectures internally serializes AVX2 gather into many μops (14-35 μops in *Zen 5*). The main downside of these heavily microcoded gathers is that it can take up a large fraction of the reorder buffer (ROB), reducing OoO execution capabilities.

I can also emperically confirm by having repeated the previous benchmark on a system with an `AMD Ryzen 7 5700` that the `uint32` LUT yields only a small speed up on *Zen 3*: 5.16 ns/pt → 4.98 ns/pt. This is in line with the expectation that the gather is heavily microcoded (23 μops). In fact, LLVM doesn't even bother with `vpgather` on *Zen 3*, it just emits a scalar loop with `load i32` instructions.

# Bit-Plane Tiling
The slow-down from `nbits=16` to `nbits=32` is now large (×4.7): 3.6 ns/pt → 17 ns/pt, while the amount of work only doubles. In practice, when the instruction count per element (point coordinates) increases, it is expected to see a performance regression as it leave less room to overlap work across elements. However, this alone cannot explain this slow down, suggesting that there are other factors at play. Lower instruction counts enables (a speed up from) full loop unrolling, vectorization, constant folding, dead code elimination, and more independent instructions for better ILP.

Note that the per-element increased instruction is not due to element loads from memory. That is because streaming accesses rely on hardware prefetchers rather than solely on ROB depth to issue future memory requests. Specifically, the prefetchers generate speculative cache-line fills ahead of demand loads. Without hardware prefetching, the slow down from poor overlap would be catastrophic, as the latency from memory accesses would be severly exposed.

### Reducing instruction count with bit-plane tiling
So far the kernels process one coordinate bit level at a time. However, there is no inherent limitation preventing us from processing multiple bit levels at a time. In fact, it is precisely what the `fast_hilbert` Rust implementation does: process 3 bit levels at a time. 

With HilbertSFC we take this a step further and process 4 bit levels (`TILE_SIZE=4`) at a time, given that it is a natural factor (power of 2). You could go even higher and process even more bits at a time. However, it is important to note that the LUT size grows exponentially with the number of bits processed at a time. In particular:
```
TABLE_SIZE = NUM_STATES × 2^(N_DIM × TILE_SIZE)
```
For the 2D case with 4 bits at a time, we have:
```
4 × 2^(2 × 4) = 4 × 256 = 1024 entries
```
which still comfortably fits in L1D cache.

We may also observe that the number of states remains constant with respect to the number of bits processed at a time due to the Markovian nature of the traversal.

The idea of this bit-plane tiling approach is to pre-compute all transitions and full `q-b` bijection for 4 bits at a time. The full table can be build by enumerating all possible input combinations of `state`, `x`, and `y` and applying our existing Hilbert encoder to build the 8 `q`-bits and the final `next_state`.

We can chose the format of the input and outputs of these tables completely as we desire as long as the original data is preserved. For example, for the coordinate bits `b`, there is no reason to interleave the `x` and `y` bits: `b = xyxyxyxy`. It is much simpler to just store them in a non-interleaved format: `b = xxxxyyyy`. This way, we can directly extract and pack the `x` and `y` bits with one shift and one mask each. For `q` we want the bits to be dense and in order of the final index, such that we can append or directly shift them into place.

8 bits are needed for `q` (4 levels × 2 bits per level), and 2 bits for the next state, so we need 10 bits total to store the output of each entry. To keep a state-independent gather, we can pack 4 entries together, which gives us 40 bits. This fits comfortably in a 64-bit value. Particularly we pack the `q`-bits in the upper 8 bits, and the 2-bit next state in the lower 4th and 5th bit:
```
qqqqqqqq 00ss0000
```
There is a specific reason to go with this layout, which will become clear when we write the encoding loop.

In this notebook, we will generalize the pattern to support any `TILE_SIZE` up to 5. We adapt the code in `scripts/hilbertsfc_gen/lut_2d4b.py`:

In [55]:
# Largely copied from `scripts/hilbertsfc_gen/lut_2d4b.py`

#     This builds two 1D lookup tables, indexed by a single (2*TILE_SIZE)-bit value:

#     - ``LUT_2D4B_B_QS[b]`` where ``b = (x << TILE_SIZE) | y``.
#     - ``LUT_2D4B_Q_BS[q]`` where ``q`` is n quadrants packed
#       as n symbols of 2 bits each (total 2*TILE_SIZE bits).

#     Each table entry is a ``uint64`` containing 4 independent 'lanes', one per
#     FSM state, with each lane occupying 16 bits. The lane selection is done by
#     shifting by ``state_shift = state << 4`` (i.e. 0, 16, 32, 48). This enables
#     state-independent lookup (gather) into the table, with state selection
#     performed afterward via a relatively cheap variable register shift rather
#     than a state-dependent gather.

#     Packed layout (lanes are states 0..3, low-to-high):

#                               uint64
#         +-----------------------------------------------+
#         | qqqq qs00 | qqqq qs00 | qqqq qs00 | qqqq qs00 |
#         |     3     |     2     |     1     |     0     |  <- lane = state
#         +-----------------------------------------------+

#         qqqqq: 10 bits, five 2-bit quadrants, MSB-first, left aligned
#         s00: 6 bits, where s is the 2-bit next_state stored as (next_state<<4)

#     For the decode table, the same pattern applies, except the upper 10 bits is
#     ``bbbbb`` (the 10-bit packed input bits) instead of ``qqqqq``.

N_STATES = 4


def generate_luts_2dnb_compacted(tile_size):
    b_entries = 1 << (2 * tile_size)  # max 10-bit bb (xxxxxyyyyy)
    lut_2dnb_b_qs = np.zeros(b_entries, dtype=np.uint64)
    lut_2dnb_q_bs = np.zeros(b_entries, dtype=np.uint64)

    for i in range(b_entries):
        for state in range(N_STATES):
            s_next = state
            qq = 0
            bb = i

            # Simulate tile_size iterations of Hilbert encoding
            for bit in reversed(range(tile_size)):
                x = (bb >> (tile_size + bit)) & 1
                y = (bb >> bit) & 1
                b = (x << 1) | y
                q = int(LUT_2D_O_B[s_next, b])
                s_next = int(LUT_2D_T_B[s_next, b])
                qq = (qq << 2) | q

            s_lsh4 = state << 4
            s_next_shifted = (s_next << 4) << s_lsh4
            qb_shift = s_lsh4 + 16 - 2 * tile_size
            qq_shifted = qq << qb_shift
            bb_shifted = bb << qb_shift

            # Pack encoding
            lut_2dnb_b_qs[bb] |= qq_shifted | s_next_shifted
            # Pack decoding
            lut_2dnb_q_bs[qq] |= bb_shifted | s_next_shifted

    return lut_2dnb_b_qs, lut_2dnb_q_bs

### Tiled kernels: when `TILE_SIZE` does not divide `nbits`

The structure of the encoding algorithm for the tiled version is largely similar to the bit-by-bit version. The loop now decrements the bit level by `TILE_SIZE`, and the masks are wider to extract multiple bits at a time. The main question remains how we should handle coordinates when `nbits` is not a multiple of `TILE_SIZE`.

There are several ways to do this. For example, you can start with full tiles, and then for the final tile simply pad the remaining coordinate bits with zeros and mask the output `q`-bits accordingly.

Here we take a different approach where we start with a potential partial tile, and then continue with full tiles. Especially for cases such as `TILE_SIZE=4` this allows for more opportunities to constant fold.
Consider, `TILE_SIZE=4`, with coordinate bits selected as:
```
b_x = (x >> bit) & 0xF
...
b = b_x << 4 | b_y
```
For `bit=4`, the right shift and left shift cancel algebraically, and can thus be eliminated. An optimizing compiler eliminates both operations and instead folds the shift into the mask, effectively transforming the two expression into `(x & 0xF0) | b_y`.

Under the scheme where the final tile ends at `bit = 0`, any excess high-order bits in the initial tile are masked out. This forces those coordinate bits to zero, which is equivalent to a lower `nbits`. As shown earlier, when `b = 00`, the state alternates deterministically between 0 and 1. Consequently, the only remaining requirement is to determine whether the first processed tile begins at an even or odd bit position, since that parity fully determines the initial state.

In [ ]:
def build_tiled_impl(tile_size, return_kernels=False):
    lut_2dnb_b_qs, lut_2dnb_q_bs = generate_luts_2dnb_compacted(tile_size)

    ts = tile_size
    qb_shift = 16 - 2 * ts

    coord_mask = np.uint64((1 << ts) - 1)  # Mask for ts bits of x or y
    q_mask = np.uint64((1 << (ts << 1)) - 1)  # Mask for q (ts quadrants of 2 bits each)
    x_mask = coord_mask << (
        qb_shift + ts
    )  # In-place mask of LUT x bits in the decode table
    y_mask = coord_mask << qb_shift  # In-place mask of LUT y bits in the decode table

    @nb.njit(inline="always")
    def hilbert_encode_2d_tiled(x, y, nbits):
        idx = 0

        start_bit = (nbits - 1) // ts * ts  # Round down to nearest multiple of ts
        drop_bits = start_bit - nbits + ts
        if drop_bits > 0:  # Conditional compilation
            mask = np.uint64((1 << nbits) - 1)
            x &= mask  # Free (merges with first 0xF mask in loop)
            y &= mask

        # Start state is either 0 or 1 based on nbits parity
        state = ((start_bit + ts) % 2) << 4

        # Process bits from MSB to LSB, i.e, (..., 8, 4, 0)
        for bit in range(start_bit, -1, -ts):
            b = ((x >> bit) & coord_mask) << ts | ((y >> bit) & coord_mask)
            qs = lut_2dnb_b_qs[b] >> state  # state is << 4: (0, 16, 32, 48)
            q = (qs & 0xFFC0) >> qb_shift  # >> qb_shift to get quadrant (free lshr)

            idx |= q << (bit << 1)  # Append ts quadrant bits to idx
            state = qs & 0x30  # advance state

        return np.uint64(idx)

    @nb.njit(inline="always")
    def hilbert_decode_2d_tiled(idx, nbits):
        x = y = 0

        start_bit = (nbits - 1) // ts * ts
        # Mask higher than nbits, use fact that q = 00, s=00 -> b=00
        drop_bits = start_bit - nbits + 4
        if drop_bits > 0:
            idx &= np.uint64((1 << (nbits << 1)) - 1)  # Merges with 0xFF mask

        # Start state is either 0 or 1 based on nbits parity
        state = ((start_bit + ts) % 2) << 4

        # Process quadrants from MSB to LSB, i.e, (..., 16, 8, 0)
        for bit in range(start_bit, -1, -ts):
            q = (idx >> (bit << 1)) & q_mask  # Extract the Hilbert quadrant (2*ts bits)
            bs = lut_2dnb_q_bs[q] >> state  # Also eliminates zext when idx uint64

            b_x = (bs & x_mask) >> (qb_shift + ts)  # Free lshr
            b_y = (bs & y_mask) >> qb_shift
            x |= b_x << bit  # Append 4 bits to x
            y |= b_y << bit  # Append 4 bits to y

            state = bs & 0x30  # Advance state

        return x, y

    def enc(xs, ys, out, *, nbits):
        batch_impl = build_hilbert_encode_2d_batch_impl(hilbert_encode_2d_tiled, nbits)
        return batch_impl(xs, ys, out)

    def dec(idx, out_xs, out_ys, *, nbits):
        batch_impl = build_hilbert_decode_2d_batch_impl(hilbert_decode_2d_tiled, nbits)
        return batch_impl(idx, out_xs, out_ys)

    if return_kernels:
        return hilbert_encode_2d_tiled, hilbert_decode_2d_tiled
    return HilbertImplementation(
        name=f"hilbert_2d{tile_size}b",
        encode=enc,
        decode=dec,
    )


hilbert_tiled_impls = [build_tiled_impl(tile_size) for tile_size in range(1, 6)]

In [57]:
results = benches[32].run_many(hilbert_tiled_impls, print_setup=False)  # type: ignore[reportArgumentType]


impl: hilbert_2d1b
hilbert_2d1b/encode: 105.601 ms | 21.12 ns/pt | 47.35 ± 0.91 Mpts/s | trials=19
hilbert_2d1b/decode: 125.286 ms | 25.06 ns/pt | 39.91 ± 0.69 Mpts/s | trials=17

impl: hilbert_2d2b
hilbert_2d2b/encode: 22.166 ms | 4.43 ns/pt | 225.57 ± 9.70 Mpts/s | trials=91
hilbert_2d2b/decode: 24.044 ms | 4.81 ns/pt | 207.95 ± 24.07 Mpts/s | trials=79

impl: hilbert_2d3b
hilbert_2d3b/encode: 14.039 ms | 2.81 ns/pt | 356.16 ± 39.00 Mpts/s | trials=133
hilbert_2d3b/decode: 13.710 ms | 2.74 ns/pt | 364.68 ± 8.49 Mpts/s | trials=145

impl: hilbert_2d4b
hilbert_2d4b/encode: 9.264 ms | 1.85 ns/pt | 539.73 ± 18.46 Mpts/s | trials=215
hilbert_2d4b/decode: 9.529 ms | 1.91 ns/pt | 524.74 ± 20.93 Mpts/s | trials=209

impl: hilbert_2d5b
hilbert_2d5b/encode: 9.026 ms | 1.81 ns/pt | 553.97 ± 29.44 Mpts/s | trials=221
hilbert_2d5b/decode: 9.314 ms | 1.86 ns/pt | 536.81 ± 23.14 Mpts/s | trials=213


### Tiling results (encode + decode)

The benchmarks demonstrate a massive performance gain from bit-plane tiling in both the encoder and decoder, with the most dramatic improvement occurring when moving from `TILE_SIZE = 1` to `TILE_SIZE = 2`. Although the theoretical workload is only reduced by roughly half, observed throughput increases by nearly a factor of five! As shown earlier for lower `nbits`, this disproportionate speedup is attributable to the compiler optimizations unlocked by lowering the per-element instruction count. 

With these optimizations in place, further increases in tile size deliver more moderate gains. Nevertheless, throughput still more than doubles when moving from `TILE_SIZE = 2` to `TILE_SIZE = 4`. After all, reduced per-element instruction counts has a secondary effect of enabling more overlap of work across independent elements, increasing ILP and MLP.


In [ ]:
encoder, decoder = build_tiled_impl(tile_size=4, return_kernels=True)  # type: ignore[reportTypeNotCallable]
batch_enc = build_hilbert_encode_2d_batch_impl(encoder, 32)
batch_enc(
    xs=np.array([0], dtype=np.uint32),
    ys=np.array([0], dtype=np.uint32),
    out=np.zeros(1, dtype=np.uint64),
)
text = batch_enc.inspect_llvm(batch_enc.signatures[0])
print(text)

## Tiled LLVM IR walkthrough (`TILE_SIZE=4`, `nbits=32`)

Inspection of the emitted optimized LLVM IR for `nbits=32` and `TILE_SIZE=4` reveals the optimizations that were specifically targeted with the design of the LUT and the encoding loop. The loop is fully unrolled, vectorized with 4 lanes (`<4 x i64>`), and applies aggressive constant folding.

If we go through the code in the inner loop, line by line, we can see how some of these optimizations are realized:
```
b = ((x >> bit) & coord_mask) << ts | ((y >> bit) & coord_mask)
```
`coord_mask`, `ts`, and `bit` are all compile time constants (`bit` is const because of the full unroll). The additional `ts` shift simply merges with the right shift on `x`.

```
qs = lut_2dnb_b_qs[b] >> state
```
The load becomes a vectorized gather, and importantly is independent of `state` (i.e., independent of previous 'iterations' in the inner loop). With the bit-plane unpacking also being independent, it can set out many loads in parallel, which is critical to hide its per scalar 4-5 cycle latency. In addtion, `state` is pre-shifted by 4 in the LUT (`state ∈ {0, 16, 32, 48}`), so there is no need for an additional shift to select the right `qs` pair.

```
q = (qs & 0xFFC0) >> qb_shift
idx |= q << (bit << 1)
```
Here the constant left shift of `(bit << 1)` simply merges with the constant right shift of `q` by `qb_shift` (which is `TILE_SIZE * 2 - 8`). The `AND` mask with `0xFFC0` is shifted into place before `OR`ing into the index.

```
state = qs & 0x30
```
This remains a simple `AND` with a constant, and thus keeps the `state`-depenency chain to 1 cycle per tile.




# Comparison with fast_hilbert Rust Implementation
**fast_hilbert** in *Rust* achieves 13.71 ns/pt, whereas our implementation with a similar `TILE_SIZE=3` achieves a throughput of 2.91 ns/pt.

Wherein lies this difference?

Both *Rust* and *Numba* use LLVM as their backend, so the performance is not inherently limited by the choice of language. The difference is entirely attributable to algorithmic structure and implementation strategy.

We see that **fast_hilbert** also uses a single LUT that packs both the state and output quadrant. Also the dependency chain on the `state` is kept minimal with just a single `AND` mask.

### State-dependent LUT

The key distinction is that **fast_hilbert**, unlike **HilbertSFC**, does not perform state independent lookups. Particulary, the LUT is indexed with `b | state`:
```
let index = (x_in | y_in | state.into()).as_usize();
let r = LUT_3[index]
```
As thoroughly discussed earlier, this greatly limits the potential for ILP and MLP, which is critical to achieve top performance.

### Runtime bit-skipping

To its credit, **fast_hilbert** does use a smart runtime trick where it finds the most significant non-zero bit across both `x` and `y`, allowing it to skip leading zero bit levels. When data is heavily skewed toward lower bits and no tight upper bound on nbits is known, this can substantially reduce the total amount of work.

However, with `TILE_SIZE = 3`, both coordinates must already fall within the lowest 1/8th of the grid before this optimization yields any speedup at all. To achieve a more meaningful improvement, they must lie within the lowest 1/64th of the grid, and so on. In practical terms, this means the data must be extremely concentrated within a very small spatial region to produce a significant performance gain.

Still, it might seem like an obvious improvement to skip leading zero tiles. However, this trick comes at a hefty cost, as it leads to variable loop bounds. This completely prevents the compiler to unroll the loop, constant fold the shifts, and, perhaps most importantly, to vectorize the loop.

In fact, the fixed-structure design of **HilbertSFC** is so amenable to compiler optimization that even in degenerate cases where all data precisely falls within the single lowest tile, it can still outperform **fast_hilbert** with its runtime bit-skipping strategy.

# Main takeaways

HilbertSFC's speed comes for the most part from designing a kernel structure and clever table layout that aligns with the reality of modern CPU architectures: minimize memory traffic, shorten dependency chains, and make the hot loop maximally optimizable by LLVM.

Key takeaways:

- NumPy/naively-JIT'd Skilling-style code tends to become bandwidth/latency bound due to poor locality and strided bit-slice access patterns.
- Making Hilbert traversal explicit as a finite state machine lets us precompile the geometry into tiny LUTs and keep per-point work light and within registers.
- The big wins come from breaking state-dependent load chains (state-independent LUT layout) and from bit-plane tiling (lower instruction count → unroll + constant fold + vectorize).
- "No loads" (register-packed LUT) isn't automatically faster if it lengthens the critical path with extra dependent shifts/masks.
- Runtime optimizations like leading zero bit skipping can backfire by preventing strong compiler optimizations.